# Overnight CPU experiment

Three methods × three paired seeds, 16,384 training decisions per condition. No GPU or hosted API is used. This is an exploratory inheritance pilot with personality held fixed; it cannot establish a prosociality effect. Read [the design and results review](../docs/overnight-cpu.md).

Run All initially only displays the plan and saved results. Enable the benchmark or training explicitly below. The same output directory resumes interrupted training and skips completed conditions. Keep the machine awake; the kernel must remain running.

The 4,096-demonstration readiness check has not yet beaten random survival. Start with seed 11, inspect learning and reproduction, then set `SEEDS = None` to expand to the full nine-run matrix.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
CONFIG = "configs/overnight-cpu.yaml"
OUTPUT = "runs/overnight-cpu"
HOURS = 10
SEEDS = [11]  # First paired cohort; use None to expand to all three seeds.
RUN_BENCHMARK = False
RUN_TRAINING = False

def launch(mode):
    cmd = [sys.executable, "-u", "-m", "millstlabs.batch", "--config", CONFIG,
           "--output", OUTPUT, "--mode", mode, "--hours", str(HOURS)]
    if SEEDS is not None:
        cmd += ["--seed", *map(str, SEEDS)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        try:
            for line in proc.stdout:
                print(line, end="", flush=True)
            if proc.wait():
                raise RuntimeError(f"Runner exited with code {proc.returncode}; inspect output above.")
        except KeyboardInterrupt:
            proc.terminate()
            proc.wait()
            print("Stopped. The next invocation resumes the last saved checkpoint.")
            raise
launch("plan")

## Measure this machine

Set `RUN_BENCHMARK = True` in the settings cell to measure CPU throughput and peak memory (about two minutes on the development Mac). The benchmark stops at extinction or 128 ticks. Its timing excludes warm starts, evaluation, model loading, and inheritance probes. Do not multiply a four-tick smoke run into an overnight estimate.

In [ ]:
if RUN_BENCHMARK:
    launch("benchmark")
else:
    print("Benchmark disabled; set RUN_BENCHMARK=True to measure this machine.")

## Execute or resume

Set `RUN_TRAINING = True` in the settings cell, then rerun it and this cell. The first time includes a 4,096-demonstration shared warm start for each seed. Runs execute serially with four CPU threads. The ten-hour limit is soft: an in-progress warm start, evaluation, or optimizer update completes before pausing. Rerun with the same config/output to continue.

For an unattended Mac run, the terminal alternative is `caffeinate -i bash scripts/overnight.sh --mode run --hours 10 --seed 11`. From the repository root on either Mac or Codespaces, `bash scripts/overnight.sh --mode run --hours 10 --seed 11` runs the same experiment.

In [ ]:
if RUN_TRAINING:
    launch("run")
else:
    print("Training disabled. Set RUN_TRAINING=True when ready for the overnight run.")

## Review saved results

Restricted mean lifetime is capped at 512 ticks. Each line below is one independently trained seed/condition. Initial-to-final changes use only their common evaluation maps. Final method comparisons use all four maps. Three training seeds are exploratory; map episodes are not additional independent training replicates. No births means the run has not tested inheritance.

In [ ]:
from millstlabs.config import load_config

root = ROOT / OUTPUT
if (root / "config.yaml").exists():
    saved = load_config(root / "config.yaml")
    assert saved.digest() == load_config(CONFIG).digest(), "Output belongs to another configuration"
    # Read live logs directly; leave status/analysis generation to the runner after it exits.
    rows = []
    for directory in sorted(root.glob("prosocial-*-s*")):
        evaluation = directory / "evaluation.jsonl"
        if evaluation.exists():
            rows += [dict(run=directory.name, **json.loads(line)) for line in evaluation.read_text().splitlines()]
    if rows:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 4))
        for name in sorted({r["run"] for r in rows}):
            points = sorted([r for r in rows if r["run"] == name], key=lambda r: r["checkpoint"])
            ax.plot([r["checkpoint"] for r in points], [r["restricted_mean_lifetime"] for r in points], marker="o", label=name)
        ax.set(xlabel="Training decision checkpoint", ylabel="Restricted mean lifetime (ticks)", ylim=(0, 512))
        ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        plt.show()
    else:
        print("No evaluation saved yet.")
    status = root / "status.json"
    if status.exists():
        result = json.loads(status.read_text())
        print(f"At last runner exit: {result['complete']}/{result['total']} conditions complete")
        for row in result["runs"]:
            print(row["method"], row["seed"], row["status"], row.get("final_lifetime"), row.get("lifetime_change_on_common_maps"))
else:
    print("No overnight results yet.")